# 🎬 English → Malayalam Video Dubbing
### Free · runs in your browser · no loss in video quality

This notebook takes an **English video from your Google Drive** and makes a Malayalam version:

| You get | Details |
|---|---|
| 🗣️ Malayalam voice | Male or female neural voice, timed to each sentence |
| 🎵 Background kept | Music and sound effects stay under the new voice |
| 🔁 English kept | Original English audio as a second track you can switch to |
| 💬 Malayalam subtitles | Can be turned on and off (not burned into the picture) |
| 🎞️ Same picture | The video picture is copied as it is, with no re-encoding |

Everything is saved to **My Drive → Malayalam_Dubbing**.

---

## ✋ Before you start (one time only, about 5 minutes)

**1. Turn on the free GPU**
Menu **Runtime → Change runtime type → T4 GPU → Save**.

**2. Get a free Hugging Face token** (needed for the translation model)
1. Sign up at [huggingface.co](https://huggingface.co/join) (you can use Google).
2. Open [ai4bharat/indictrans2-en-indic-1B](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B) and click **Agree and access repository**.
3. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → **Create new token** → choose the **Read** tab → name it `colab` → **Create token** → copy it (starts with `hf_`).

**3. Get a free Gemini API key** (optional, makes the Malayalam sound natural and spoken)
1. Open [aistudio.google.com](https://aistudio.google.com) and sign in with Google.
2. Click **Get API key → Create API key** and copy it. No credit card needed.

**4. Save both keys in Colab Secrets** (recommended, so they never appear in the notebook)
1. Click the **🔑 key icon** in the left sidebar.
2. **Add new secret** → Name `HF_TOKEN` → paste your Hugging Face token → turn on **Notebook access**.
3. **Add new secret** → Name `GEMINI_API_KEY` → paste your Gemini key → turn on **Notebook access**.

> You can also paste the keys into the Step 2 form instead. If you do, **never share or upload that copy of the notebook**, because anyone could read your keys.

---

## ▶️ How to run

1. **Run Step 1** (press ▶ on it). It installs everything and then **restarts the session by itself**. A message like *"session restarted"* or *"session crashed"* is expected.
2. **Fill in Step 2**: paste your video's Google Drive link.
3. **Click inside Step 2 and press `Ctrl + F10`** (or menu **Runtime → Run cell and below**). That runs Step 2 to Step 9 in one go.
4. Click **Allow / Continue** on the Google sign-in pop-ups (there are two).
5. Wait about **15–20 minutes** for a 6-minute video. Then find your video in **My Drive → Malayalam_Dubbing**.

**Red text while installing** (dependency conflict warnings) is normal and can be ignored.
**Every step remembers its work.** If something stops halfway, just run that step again; finished steps are skipped automatically.

In [ ]:
# @title Step 1 – Install everything (once per session, about 3–5 minutes) { display-mode: "form" }
# @markdown Just press ▶. When it finishes, the session **restarts by itself**. That is expected. Then go to Step 2.
import os, sys, subprocess, time

DUBKIT = r'''"""Shared helpers for the English -> Malayalam dubbing notebook.
Written to /content/dubkit.py by Step 1. Every step does `from dubkit import *`,
so each step can be re-run on its own, even after a session restart."""
import os, re, json, glob, ctypes, site, subprocess, shutil, time

# Use PyTorch only (avoids the TensorFlow / protobuf clash in Colab)
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np

WORK = '/content/work'                       # fast local scratch space (lost when Colab disconnects)
DRIVE_ROOT = '/content/drive/MyDrive/Malayalam_Dubbing'
STATE = os.path.join(WORK, 'state.json')
SR = 44100
SETUP_MARK = '/content/.dub_setup_done'
os.makedirs(WORK, exist_ok=True)

VOICES = {'Male (Midhun)': 'ml-IN-MidhunNeural', 'Female (Sobhana)': 'ml-IN-SobhanaNeural'}


# ---------- small utilities ----------
def ok(msg):
    print('✅ ' + msg)


def stop(msg):
    """Stop the cell with a clear, friendly message (no long traceback)."""
    raise SystemExit('❌ ' + msg)


def require_setup():
    if not os.path.exists(SETUP_MARK):
        stop('Run Step 1 first (it installs the tools).')


# ---------- subtitles ----------
def ts(t):
    ms = int(round(max(t, 0) * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f'{h:02}:{m:02}:{s:02},{ms:03}'


def write_srt(segs, path):
    with open(path, 'w', encoding='utf-8') as f:
        for i, (a, b, t) in enumerate(segs, 1):
            f.write(f'{i}\n{ts(a)} --> {ts(b)}\n{t}\n\n')


def _sec(s):
    s = s.strip().replace('.', ',')
    h, m, rest = s.split(':')
    sec, ms = (rest.split(',') + ['0'])[:2]
    return int(h) * 3600 + int(m) * 60 + int(sec) + int(ms.ljust(3, '0')[:3]) / 1000


def read_srt(path):
    """Read an .srt file into a list of (start_sec, end_sec, text). Tolerant of hand edits."""
    text = open(path, encoding='utf-8-sig').read().replace('\r\n', '\n').replace('\r', '\n')
    segs = []
    for block in re.split(r'\n\s*\n', text.strip()):
        lines = block.strip().split('\n')
        k = next((i for i, l in enumerate(lines) if '-->' in l), None)
        if k is None:
            continue
        a, b = lines[k].split('-->')
        segs.append((_sec(a), _sec(b.split()[0]), ' '.join(x.strip() for x in lines[k + 1:]).strip()))
    return segs


# ---------- remembered settings (so every step can run on its own) ----------
def load_state(required=True):
    if os.path.exists(STATE):
        with open(STATE) as f:
            return json.load(f)
    if required:
        stop('No video selected yet. Fill in Step 2 and run it first.')
    return {}


def save_state(**kw):
    s = load_state(required=False)
    s.update(kw)
    with open(STATE, 'w') as f:
        json.dump(s, f, indent=2)
    return s


def out_file(s, suffix):
    """A file in the video's output folder on Google Drive."""
    return os.path.join(s['out'], s['name'] + suffix)


def work_file(s, suffix):
    """A file in the local work folder."""
    return os.path.join(WORK, s['key'] + suffix)


def mount_drive():
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')


def get_secret(value, name):
    """Use the typed value, else the Colab Secret with this name (key icon on the left)."""
    value = (value or '').strip()
    if value:
        return value
    try:
        from google.colab import userdata
        return (userdata.get(name) or '').strip()
    except Exception:
        return ''


# ---------- GPU helpers ----------
def load_cuda12():
    """faster-whisper needs the CUDA 12 cuBLAS/cuDNN libraries; load them from pip wheels."""
    libs = []
    for sp in site.getsitepackages():
        for sub in ('cublas', 'cudnn'):
            libs += glob.glob(os.path.join(sp, 'nvidia', sub, 'lib', '*.so*'))
    pending = sorted(set(libs))
    for _ in range(5):
        failed = []
        for lib in pending:
            try:
                ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                failed.append(lib)
        if not failed:
            break
        pending = failed


def free_gpu():
    import gc
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass


# ---------- audio ----------
def run_ffmpeg(args):
    r = subprocess.run(['ffmpeg', '-y', '-nostdin', '-loglevel', 'error'] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('ffmpeg failed:\n' + r.stderr[-1500:])


def probe(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_streams', '-show_format', '-of', 'json', path],
                       capture_output=True, text=True)
    return json.loads(r.stdout or '{}')


def pcm(path, sr=SR, ch=2, speed=1.0):
    """Decode any audio/video file to float32 samples (optionally sped up)."""
    cmd = ['ffmpeg', '-nostdin', '-loglevel', 'error', '-i', path]
    if speed > 1.0:
        cmd += ['-filter:a', f'atempo={min(speed, 2.0):.3f}']
    cmd += ['-f', 'f32le', '-ac', str(ch), '-ar', str(sr), '-']
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    a = np.frombuffer(raw, np.float32)
    return a.reshape(-1, ch) if ch > 1 else a


def write_wav(path, arr, sr=SR):
    from scipy.io import wavfile
    wavfile.write(path, sr, (np.clip(arr, -1, 1) * 32767).astype(np.int16))


def read_wav(path):
    from scipy.io import wavfile
    _, a = wavfile.read(path)
    return a.astype(np.float32) / 32768
'''
with open('/content/dubkit.py', 'w') as f:
    f.write(DUBKIT)

MARK = '/content/.dub_setup_done'
if os.path.exists(MARK):
    print('✅ Setup is already done for this session. Go to Step 2.')
else:
    gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
    if 'GPU' not in gpu:
        raise SystemExit('❌ No GPU. Go to Runtime > Change runtime type > T4 GPU > Save, then run Step 1 again.')
    print('GPU found:', gpu.strip().split(' (')[0])

    def pip(*args):
        r = subprocess.run([sys.executable, '-m', 'pip', '-q'] + list(args), capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:], r.stderr[-3000:])
            raise SystemExit('❌ Install failed (details above). Run Step 1 again. If it keeps failing, open an issue on GitHub.')

    print('1/3 Removing TensorFlow (this notebook does not use it, and it clashes with the other tools)...')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'tensorflow', 'tf-keras',
                    'tensorflow-text', 'tensorflow-hub', 'tensorflow-decision-forests'], capture_output=True)

    print('2/3 Installing the free tools: Whisper, IndicTrans2, Demucs, edge-tts, Gemini (2-4 minutes, please wait)...')
    pip('install',
        'transformers>=4.51,<5',          # IndicTrans2 needs version 4
        'faster-whisper',                 # English speech -> text
        'IndicTransToolkit', 'sentencepiece', 'sacremoses',   # English -> Malayalam
        'google-genai',                   # optional natural-Malayalam polish
        'demucs',                         # separates music from voices
        'edge-tts',                       # Malayalam voice
        'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*')       # GPU libraries Whisper needs

    print('3/3 Checking ffmpeg...')
    if subprocess.run(['which', 'ffmpeg'], capture_output=True).returncode != 0:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], capture_output=True)

    open(MARK, 'w').write('ok')
    print('\n✅ All installed. The session will now RESTART BY ITSELF (this is normal).')
    print('➡️  When it reconnects, fill in Step 2, click inside it and press Ctrl+F10.')
    time.sleep(3)
    try:
        get_ipython().kernel.do_shutdown(True)
    except Exception:
        os.kill(os.getpid(), 9)

In [ ]:
# @title Step 2 – Your video and settings
# @markdown **Paste your video's Google Drive link.** In Drive: right-click the video → **Share → Copy link**.
# @markdown (A path like `/content/drive/MyDrive/folder/video.mp4` also works.)
VIDEO_LINK = ""  # @param {type:"string"}
# @markdown ---
# @markdown **Keys:** leave empty if you saved them in Colab Secrets (🔑 icon on the left). Otherwise paste them here.
HF_TOKEN = ""  # @param {type:"string"}
GEMINI_API_KEY = ""  # @param {type:"string"}
# @markdown ---
# @markdown **Voice and sound**
VOICE = "Male (Midhun)"  # @param ["Male (Midhun)", "Female (Sobhana)"]
MUSIC_VOLUME = 0.6  # @param {type:"slider", min:0, max:1, step:0.05}
KEEP_ENGLISH_AUDIO = True  # @param {type:"boolean"}
NATURAL_MALAYALAM_WITH_GEMINI = True  # @param {type:"boolean"}
# @markdown ---
# @markdown Then click inside this cell and press **Ctrl + F10** to run all the remaining steps.

import io, sys
sys.path.insert(0, '/content')
from dubkit import *
require_setup()

link = VIDEO_LINK.strip().strip('"').strip("'")
if not link:
    stop('Paste your video link into VIDEO_LINK above, then run this cell again.')

mount_drive()

if link.startswith('/content/'):
    if not os.path.isfile(link):
        stop(f'File not found: {link}')
    video, filename = link, os.path.basename(link)
else:
    m = re.search(r'/d/([\w-]+)', link) or re.search(r'[?&]id=([\w-]+)', link)
    if not m:
        stop('That does not look like a Google Drive file link. In Drive: right-click the video → Share → Copy link.')
    file_id = m.group(1)
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    svc = build('drive', 'v3')
    try:
        meta = svc.files().get(fileId=file_id, fields='name,size,mimeType', supportsAllDrives=True).execute()
    except Exception as e:
        stop('Could not open that file. Make sure it is in (or shared with) the Google account you just signed in with.\n'
             f'Details: {str(e)[:200]}')
    if not meta.get('mimeType', '').startswith('video/') and not meta['name'].lower().endswith(
            ('.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v')):
        stop(f'"{meta["name"]}" does not look like a video file.')
    filename = meta['name']
    video = os.path.join(WORK, 'source_' + filename.replace('/', '_'))
    size = int(meta.get('size', 0))
    if os.path.exists(video) and os.path.getsize(video) == size:
        ok('Video already downloaded.')
    else:
        print(f'Downloading "{filename}" ({size / 1e6:.0f} MB). This is an exact copy, so there is no quality loss.')
        req = svc.files().get_media(fileId=file_id, supportsAllDrives=True)
        with io.FileIO(video, 'wb') as fh:
            dl = MediaIoBaseDownload(fh, req, chunksize=64 * 1024 * 1024)
            finished, shown = False, -1
            while not finished:
                status, finished = dl.next_chunk()
                if status and int(status.progress() * 10) != shown:
                    shown = int(status.progress() * 10)
                    print(f'  {shown * 10}%')
        ok('Downloaded.')

info = probe(video)
if not any(st.get('codec_type') == 'video' for st in info.get('streams', [])):
    stop('No video track found in that file.')
if not any(st.get('codec_type') == 'audio' for st in info.get('streams', [])):
    stop('This video has no sound, so there is nothing to translate.')
minutes = float(info.get('format', {}).get('duration', 0)) / 60

name = os.path.splitext(filename)[0].replace('/', '_')
key = re.sub(r'[^\w-]+', '_', name)[:60]
out = os.path.join(DRIVE_ROOT, name + '_malayalam')
os.makedirs(out, exist_ok=True)

s = save_state(video=video, name=name, key=key, out=out,
               hf_token=get_secret(HF_TOKEN, 'HF_TOKEN'),
               gemini_key=get_secret(GEMINI_API_KEY, 'GEMINI_API_KEY'),
               voice=VOICES[VOICE], music=float(MUSIC_VOLUME),
               keep_english=bool(KEEP_ENGLISH_AUDIO), use_gemini=bool(NATURAL_MALAYALAM_WITH_GEMINI))

if not os.path.exists(work_file(s, '_audio.wav')):
    run_ffmpeg(['-i', video, '-vn', '-ac', '2', '-ar', str(SR), work_file(s, '_audio.wav')])

ok(f'Video: {filename} ({minutes:.1f} minutes)')
print('   Output folder: My Drive > Malayalam_Dubbing >', os.path.basename(out))
print('   Hugging Face token:', 'found' if s['hf_token'] else 'MISSING (Step 4 needs it, see the instructions at the top)')
if s['use_gemini']:
    print('   Gemini key:', 'found' if s['gemini_key'] else 'not found, so Step 5 will keep the plain translation')
print('   Voice:', VOICE, '| Music volume:', MUSIC_VOLUME, '| Keep English audio:', KEEP_ENGLISH_AUDIO)

In [ ]:
# @title Step 3 – English transcript with Whisper (about 3 minutes)
REDO = False  # @param {type:"boolean"}
# @markdown Tick **REDO** only if you want to transcribe again from scratch.
import sys
sys.path.insert(0, '/content')
from dubkit import *
require_setup()
s = load_state()
en_srt = out_file(s, '_english.srt')

if os.path.exists(en_srt) and not REDO:
    EN = read_srt(en_srt)
    ok(f'Already transcribed ({len(EN)} lines). Skipping.')
else:
    load_cuda12()
    from faster_whisper import WhisperModel
    audio16k = pcm(work_file(s, '_audio.wav'), sr=16000, ch=1)   # read with ffmpeg (avoids a PyAV bug)
    total = len(audio16k) / 16000
    print(f'Transcribing {total / 60:.1f} minutes of English speech...')
    wm = WhisperModel('large-v3', device='cuda', compute_type='float16')
    segments, _ = wm.transcribe(audio16k, language='en', vad_filter=True, beam_size=5,
                                condition_on_previous_text=False)
    EN, shown = [], -1
    for seg in segments:
        if seg.text.strip():
            EN.append((seg.start, seg.end, seg.text.strip()))
        if int(seg.end / total * 10) != shown:
            shown = int(seg.end / total * 10)
            print(f'  {min(shown * 10, 100)}%')
    if not EN:
        stop('No English speech was found in this video.')
    write_srt(EN, en_srt)
    del wm
    free_gpu()
    ok(f'{len(EN)} lines transcribed and saved.')

for a, b, t in EN[:5]:
    print(ts(a)[3:8], t)

In [ ]:
# @title Step 4 – Translate to Malayalam with IndicTrans2 (about 3 minutes the first time)
REDO = False  # @param {type:"boolean"}
import sys
sys.path.insert(0, '/content')
from dubkit import *
require_setup()
s = load_state()
EN = read_srt(out_file(s, '_english.srt'))
it_srt = out_file(s, '_malayalam_indictrans.srt')

if os.path.exists(it_srt) and not REDO:
    ok(f'Already translated ({len(read_srt(it_srt))} lines). Skipping.')
else:
    if not s.get('hf_token'):
        stop('Hugging Face token missing. Add it as the Colab Secret HF_TOKEN (or paste it in Step 2), '
             'run Step 2 again, then this step.')
    import torch
    from huggingface_hub import login
    login(token=s['hf_token'], add_to_git_credential=False)
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    try:
        from IndicTransToolkit.processor import IndicProcessor
    except ImportError:
        from IndicTransToolkit import IndicProcessor

    MID = 'ai4bharat/indictrans2-en-indic-1B'
    try:
        tok = AutoTokenizer.from_pretrained(MID, trust_remote_code=True)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(MID, trust_remote_code=True,
                                                    torch_dtype=torch.float16).to('cuda').eval()
    except Exception as e:
        if any(x in str(e) for x in ('gated', '401', '403', 'access')):
            stop('No access to the translation model. Open https://huggingface.co/ai4bharat/indictrans2-en-indic-1B, '
                 'click "Agree and access repository" with the same account as your token, then run this step again.')
        raise
    ip = IndicProcessor(inference=True)

    src = [t for _, _, t in EN]
    ML_TEXT = []
    for i in range(0, len(src), 16):
        batch = ip.preprocess_batch(src[i:i + 16], src_lang='eng_Latn', tgt_lang='mal_Mlym')
        x = tok(batch, truncation=True, padding='longest', return_tensors='pt').to('cuda')
        with torch.no_grad():
            g = mdl.generate(**x, max_length=256, num_beams=5, use_cache=False)
        out = tok.batch_decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        ML_TEXT += ip.postprocess_batch(out, lang='mal_Mlym')
        print(f'  Translated {min(i + 16, len(src))} of {len(src)}')

    write_srt([(a, b, t) for (a, b, _), t in zip(EN, ML_TEXT)], it_srt)
    del mdl
    free_gpu()
    ok('Malayalam translation saved.')

for (a, b, e), (_, _, m) in list(zip(EN, read_srt(it_srt)))[:5]:
    print(e, '\n  ->', m)

In [ ]:
# @title Step 5 – Make the Malayalam sound natural with Gemini (optional, about 1 minute)
REDO = False  # @param {type:"boolean"}
# @markdown ⚠️ **REDO** rewrites the subtitle file, which **removes any fixes you made in Step 6**.
import sys
sys.path.insert(0, '/content')
from dubkit import *
require_setup()
s = load_state()
EN = read_srt(out_file(s, '_english.srt'))
RAW = read_srt(out_file(s, '_malayalam_indictrans.srt'))
ml_srt = out_file(s, '_malayalam.srt')

if os.path.exists(ml_srt) and not REDO:
    ok('Malayalam subtitles already exist (they may include your fixes). Skipping.')
elif not (s.get('use_gemini') and s.get('gemini_key')):
    write_srt(RAW, ml_srt)
    ok('Using the IndicTrans2 translation as it is (Gemini switched off or no key).')
else:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=s['gemini_key'])
    model = None
    for cand in ['gemini-flash-latest', 'gemini-2.5-flash', 'gemini-2.0-flash']:
        try:
            client.models.generate_content(model=cand, contents='Reply with OK')
            model = cand
            break
        except Exception as e:
            print(f'  {cand} not available: {str(e)[:90]}')

    text = [t for _, _, t in RAW]
    if not model:
        print('Gemini could not be reached (check the key), so the plain translation is kept.')
    else:
        print('Polishing with', model)
        cfg = types.GenerateContentConfig(response_mime_type='application/json', temperature=0.4)
        B = 30
        for i in range(0, len(text), B):
            items = [{'id': j, 'seconds': round(RAW[j][1] - RAW[j][0], 1), 'english': EN[j][2], 'malayalam': text[j]}
                     for j in range(i, min(i + B, len(text)))]
            prompt = (
                'You are an expert Malayalam dubbing writer from Kerala. Rewrite each "malayalam" line so it sounds like '
                'natural, everyday spoken Malayalam that a Kerala viewer would hear in a well-dubbed video, not stiff '
                'textbook language. Rules: keep the exact meaning of the "english" line; the line must be speakable in '
                'about "seconds" seconds, so keep it short; keep names, numbers and technical terms (common English '
                'technical words may stay as Malayalis normally say them). The lines are consecutive, so keep the flow. '
                'Return ONLY a JSON list of objects with keys "id" and "malayalam".\n'
                + json.dumps(items, ensure_ascii=False))
            for attempt in range(4):
                try:
                    r = client.models.generate_content(model=model, contents=prompt, config=cfg)
                    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', (r.text or '').strip())
                    for o in json.loads(raw):
                        j = int(o.get('id', -1))
                        if i <= j < i + B and str(o.get('malayalam', '')).strip():
                            text[j] = str(o['malayalam']).strip()
                    print(f'  Polished {min(i + B, len(text))} of {len(text)}')
                    break
                except Exception as e:
                    wait = 40 if ('429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e)) else 8
                    print(f'  Waiting {wait}s and retrying ({str(e)[:70]})')
                    time.sleep(wait)
            else:
                print(f'  Kept the plain translation for lines {i + 1}-{min(i + B, len(text))}')
            time.sleep(7)   # stay inside the free-tier limit
    write_srt([(a, b, t) for (a, b, _), t in zip(RAW, text)], ml_srt)
    ok('Natural Malayalam subtitles saved.')

ML = read_srt(ml_srt)
for (_, _, raw), (_, _, new) in list(zip(RAW, ML))[:5]:
    print('Before:', raw, '\nAfter: ', new, '\n')

In [ ]:
# @title Step 6 – Check and fix the Malayalam subtitles
# @markdown Read the list below. To change a line, write it in **FIXES** (line number: new text) and run this cell again.
# @markdown Or edit the file directly: 📁 Files panel → drive → MyDrive → Malayalam_Dubbing → your folder →
# @markdown double-click the `_malayalam.srt` file → edit → **Ctrl+S** → run this cell.
# @markdown After fixing, run **Step 8 and Step 9** again (only the changed lines are re-voiced).
FIXES = {
    # 5: 'ഇവിടെ തിരുത്തിയ വാചകം',
    # 12: 'മറ്റൊരു തിരുത്ത്',
}

import sys
sys.path.insert(0, '/content')
from dubkit import *
s = load_state()
ml_srt = out_file(s, '_malayalam.srt')
ML = read_srt(ml_srt)
EN = read_srt(out_file(s, '_english.srt'))

if FIXES:
    for n, txt in FIXES.items():
        if not 1 <= n <= len(ML):
            stop(f'Line {n} does not exist (there are {len(ML)} lines).')
        a, b, _ = ML[n - 1]
        ML[n - 1] = (a, b, txt.strip())
    write_srt(ML, ml_srt)
    ok(f'Applied {len(FIXES)} fix(es) and saved.\n')

for n, (a, b, t) in enumerate(ML, 1):
    print(f'{n:>3} [{ts(a)[3:8]}]  {t}')
    if n <= len(EN):
        print(f'            EN: {EN[n - 1][2]}')
print(f'\n{len(ML)} lines. If they look fine, carry on.')

In [ ]:
# @title Step 7 – Separate the background music from the voices (about 2 minutes)
REDO = False  # @param {type:"boolean"}
import sys
sys.path.insert(0, '/content')
from dubkit import *
require_setup()
s = load_state()
bg_wav = work_file(s, '_background.wav')

if os.path.exists(bg_wav) and not REDO:
    ok('Background track already made. Skipping.')
else:
    import torch
    from demucs.pretrained import get_model
    from demucs.apply import apply_model
    model = get_model('htdemucs')
    model.cuda().eval()
    wav = torch.from_numpy(pcm(work_file(s, '_audio.wav'), sr=model.samplerate, ch=2).copy()).T
    ref = wav.mean(0)
    mean, std = ref.mean(), ref.std() + 1e-8
    print('Separating music and effects from the English voices...')
    with torch.no_grad():
        src = apply_model(model, ((wav - mean) / std)[None].cuda(), device='cuda',
                          split=True, overlap=0.25, progress=True)[0]
    src = src * std + mean
    vi = model.sources.index('vocals')
    background = (src.sum(0) - src[vi]).T.cpu().numpy()   # drums + bass + other = music and effects
    write_wav(bg_wav, background, model.samplerate)
    del model, src
    free_gpu()
    ok('Background track ready.')

from IPython.display import Audio, display
print('Preview (first 20 seconds of the background):')
display(Audio(read_wav(bg_wav)[:20 * SR].mean(1), rate=SR))

In [ ]:
# @title Step 8 – Malayalam voice, timed to each line and mixed over the music (3–6 minutes)
# @markdown Re-running is quick: lines that did not change are not re-voiced.
# @markdown To change the voice or music volume, change them in Step 2, run Step 2, then run Step 8 and Step 9.
import sys, hashlib, asyncio
sys.path.insert(0, '/content')
from dubkit import *
import edge_tts
require_setup()
s = load_state()
ML = read_srt(out_file(s, '_malayalam.srt'))
VOICE = s['voice']
bg = read_wav(work_file(s, '_background.wav'))
voice = np.zeros_like(bg)
tts_dir = os.path.join(WORK, 'tts')
os.makedirs(tts_dir, exist_ok=True)


async def speak(text, rate):
    """Make (or reuse) a voice clip for this text at this speed."""
    path = os.path.join(tts_dir, hashlib.md5(f'{VOICE}|{rate}|{text}'.encode()).hexdigest() + '.mp3')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for attempt in range(5):
        try:
            await edge_tts.Communicate(text, VOICE, rate=f'+{rate}%').save(path + '.part')
            os.replace(path + '.part', path)
            return path
        except Exception as e:
            print(f'    voice retry ({str(e)[:60]})')
            await asyncio.sleep(3 * (attempt + 1))
    raise RuntimeError('The voice service did not respond. Check the internet connection and run Step 8 again.')


fast = 0
for i, (a, b, t) in enumerate(ML):
    t = t.strip()
    if not t:
        continue
    nxt = ML[i + 1][0] if i + 1 < len(ML) else len(bg) / SR
    room = max(nxt - a - 0.05, b - a, 0.4)            # seconds available for this line
    clip = pcm(await speak(t, 0))
    if len(clip) / SR > room:                         # too long: speak faster (max +40%)
        pct = min(int((len(clip) / SR / room - 1) * 100) + 3, 40)
        p = await speak(t, pct)
        clip = pcm(p)
        if len(clip) / SR > room:                     # still too long: compress slightly (max 1.3x)
            clip = pcm(p, speed=min(len(clip) / SR / room, 1.3))
        fast += 1
    start = int(a * SR)
    end = min(start + len(clip), len(voice))
    voice[start:end] += clip[:end - start]
    if (i + 1) % 10 == 0 or i + 1 == len(ML):
        print(f'  Voiced {i + 1} of {len(ML)}')

mix = bg * s['music'] + voice * 1.1
mix /= max(1.0, float(np.abs(mix).max()) / 0.98)      # avoid distortion
mal_wav = work_file(s, '_malayalam_audio.wav')
write_wav(mal_wav, mix)
run_ffmpeg(['-i', mal_wav, '-b:a', '192k', out_file(s, '_malayalam_audio.mp3')])
ok(f'Malayalam audio ready (lines sped up to fit: {fast}). MP3 copy saved to Drive.')

from IPython.display import Audio, display
first = int(ML[0][0] * SR) if ML else 0
print('Preview (30 seconds from the first line):')
display(Audio(mix[first:first + 30 * SR].mean(1), rate=SR))

In [ ]:
# @title Step 9 – Build the final video and save it to Drive (under 1 minute)
import sys
sys.path.insert(0, '/content')
from dubkit import *
s = load_state()
video, mal_wav, ml_srt = s['video'], work_file(s, '_malayalam_audio.wav'), out_file(s, '_malayalam.srt')
final = out_file(s, '_malayalam.mkv')
if not os.path.exists(mal_wav):
    stop('Malayalam audio not found. Run Step 7 and Step 8 first.')

has_en = s.get('keep_english', True) and any(
    st.get('codec_type') == 'audio' for st in probe(video).get('streams', []))

args = ['-i', video, '-i', mal_wav, '-i', ml_srt, '-map', '0:v:0', '-map', '1:a:0']
if has_en:
    args += ['-map', '0:a:0']
args += ['-map', '2:s:0',
         '-c:v', 'copy',                                    # picture copied as it is: no quality loss
         '-c:a:0', 'aac', '-b:a:0', '192k',
         '-metadata:s:a:0', 'language=mal', '-metadata:s:a:0', 'title=Malayalam',
         '-disposition:a:0', 'default']
if has_en:
    args += ['-c:a:1', 'copy',                              # original English audio, untouched
             '-metadata:s:a:1', 'language=eng', '-metadata:s:a:1', 'title=English',
             '-disposition:a:1', '0']
args += ['-c:s', 'srt', '-metadata:s:s:0', 'language=mal', '-metadata:s:s:0', 'title=Malayalam',
         '-disposition:s:0', 'default', final]
tmp = os.path.join(WORK, 'final_tmp.mkv')
run_ffmpeg(args[:-1] + [tmp])                               # build locally, then copy to Drive
shutil.copyfile(tmp, final)
os.remove(tmp)

ok('Done! Your Malayalam video is in Google Drive:')
print('   My Drive > Malayalam_Dubbing >', os.path.basename(s['out']), '>', os.path.basename(final))
print(f'   Size: {os.path.getsize(final) / 1e6:.0f} MB. Tracks inside:')
for st in probe(final)['streams']:
    print(f"   - {st['codec_type']:<9} {st.get('tags', {}).get('title', 'Video')}")
print('\nPlay it in VLC: Audio > Audio Track to switch language, Subtitle > Sub Track for subtitles.')

---
## 🎉 Finished. What now?

**Watch it:** download the `.mkv` from **My Drive → Malayalam_Dubbing → your folder** and open it in **[VLC](https://www.videolan.org/)** (free, on PC and phone).
- **Audio → Audio Track**: switch between Malayalam and English
- **Subtitle → Sub Track**: turn Malayalam subtitles on or off

Google Drive's own player only plays the first audio track and does not show these subtitles, so check it in VLC.

**Also in the folder:** `_english.srt`, `_malayalam.srt`, `_malayalam_indictrans.srt` (before Gemini) and `_malayalam_audio.mp3`.

| I want to… | Do this |
|---|---|
| Fix some Malayalam lines | Step 6 → then Step 8 → Step 9 |
| Change voice or music volume | Change it in Step 2 → run Step 2 → Step 8 → Step 9 |
| Dub another video | Paste the new link in Step 2 → click in Step 2 → **Ctrl + F10** (no need for Step 1 again) |
| Come back another day | Run Step 1 again (Colab forgets installs when it disconnects), then Step 2 and below. Finished work in Drive is reused. |

## 🛠️ If something goes wrong

| Message | What it means / what to do |
|---|---|
| Red *"dependency conflicts"* text in Step 1 | Normal. Ignore it. |
| *"Session crashed"* or *"restarted"* after Step 1 | Normal. Step 1 restarts the session on purpose. Go to Step 2. |
| *"unauthenticated requests to the HF Hub"* | Only a warning. Ignore it. |
| *"No access to the translation model"* | Open the IndicTrans2 page and click **Agree and access repository**, then rerun Step 4. |
| *"Hugging Face token missing"* | Add the `HF_TOKEN` secret (🔑) with Notebook access on, rerun Step 2, then Step 4. |
| Gemini *429 / RESOURCE_EXHAUSTED* | Free limit reached. The step waits and retries; worst case it keeps the plain translation. |
| *"No GPU"* | Runtime → Change runtime type → T4 GPU. The free GPU may be busy at peak times; try later. |
| Anything that says *"Run Step X first"* | Run that step, then continue. |
| Colab disconnected halfway | Run Step 1, then Step 2 and below. Finished steps are skipped. |
| Some lines sound rushed | Shorten those lines in Step 6, then rerun Step 8 and Step 9. |